In [215]:
# imports
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error
import lightgbm as lgbm

In [216]:
# Last inn data
purchase_orders = pd.read_csv("data/kernel/purchase_orders.csv", parse_dates=["delivery_date", "created_date_time", "modified_date_time"]).copy()
receivals = pd.read_csv("data/kernel/receivals.csv", parse_dates=["date_arrival"]).copy()
materials = pd.read_csv("data/extended/materials.csv").copy()
transportation = pd.read_csv("data/extended/transportation.csv").copy()
mapping = pd.read_csv("data/prediction_mapping.csv", parse_dates=["forecast_start_date", "forecast_end_date"]).copy()

receivals['date_arrival'] = pd.to_datetime(receivals['date_arrival'], utc=True).dt.tz_localize(None)

# Drop data før 2007, under covid og 29 feb 2024
purchase_orders = purchase_orders[purchase_orders["created_date_time"].dt.year > 2016]
receivals = receivals[receivals["date_arrival"].dt.year > 2016]

""" receivals = receivals[~receivals["date_arrival"].dt.year.isin([2020,2021])]
purchase_orders = purchase_orders[~purchase_orders["created_date_time"].dt.year.isin([2020,2021])] """


# Fjerner negative og slettede
purchase_orders_clean = purchase_orders[purchase_orders['quantity'] > 0]
purchase_orders_clean = purchase_orders_clean[purchase_orders_clean['status'] != 'Deleted']
receivals_clean = receivals[receivals['net_weight'] > 0]


# Bygger grunnlagstabeller

purchase_base = purchase_orders_clean[["purchase_order_id", 
                                       "created_date_time",
                                       "purchase_order_item_no",
                                       ]]

receivals_base = receivals_clean[["rm_id",
                                 "purchase_order_id",
                                 "purchase_order_item_no",
                                 "product_id",
                                 "date_arrival",
                                 "net_weight"
                                 ]]

# Slår sammen for å finne leveringsgrad
rec_ord_merged = receivals_base.merge(
    purchase_base,
    on = ["purchase_order_id", "purchase_order_item_no"],
    how = "inner"
    )


data = rec_ord_merged

count_rm_2024 = data.loc[data["created_date_time"].dt.year == 2024, "rm_id"].nunique()
print(f"Unique rm_id in 2024: {count_rm_2024}")

print(data.info())
data


Unique rm_id in 2024: 58
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 49223 entries, 0 to 49222
Data columns (total 7 columns):
 #   Column                  Non-Null Count  Dtype              
---  ------                  --------------  -----              
 0   rm_id                   49223 non-null  float64            
 1   purchase_order_id       49223 non-null  float64            
 2   purchase_order_item_no  49223 non-null  float64            
 3   product_id              49223 non-null  float64            
 4   date_arrival            49223 non-null  datetime64[ns]     
 5   net_weight              49223 non-null  float64            
 6   created_date_time       49223 non-null  datetime64[ns, UTC]
dtypes: datetime64[ns, UTC](1), datetime64[ns](1), float64(5)
memory usage: 2.6 MB
None


,rm_id,purchase_order_id,purchase_order_item_no,product_id,date_arrival,net_weight,created_date_time
0,2132.0,284465.0,10.0,91900330.0,2017-01-04 10:23:00,580.0,2017-01-04 12:52:00+00:00
1,2130.0,284431.0,10.0,91900143.0,2017-01-04 15:21:00,22462.0,2017-01-03 08:16:58+00:00
2,2130.0,284536.0,10.0,91900143.0,2017-01-05 12:06:00,20936.0,2017-01-10 08:30:17+00:00
3,2130.0,284431.0,10.0,91900143.0,2017-01-09 10:54:00,13676.0,2017-01-03 08:16:58+00:00
4,2131.0,284431.0,20.0,91900302.0,2017-01-09 15:32:00,7011.0,2017-01-03 08:16:58+00:00
...,...,...,...,...,...,...,...
49218,3421.0,328740.0,30.0,91900160.0,2024-12-19 10:26:00,1220.0,2024-08-22 07:50:43+00:00
49219,2145.0,328740.0,20.0,91900146.0,2024-12-19 10:26:00,850.0,2024-08-22 07:50:42+00:00
49220,3865.0,328740.0,10.0,91900143.0,2024-12-19 10:26:00,14170.0,2024-08-22 07:50:42+00:00
49221,3901.0,330334.0,10.0,91901440.0,2024-12-19 10:54:00,15020.0,2024-11-15 12:11:38+00:00


In [217]:
# Fjerner stock_location 'DELETED' i materials
materials['product_id'] = pd.to_numeric(materials['product_id'], errors='coerce')
materials['rm_id'] = pd.to_numeric(materials['rm_id'], errors='coerce')

active_materials = materials[~materials['stock_location'].astype(str).str.upper().str.contains('DELETED')].copy()

# 2) Tillatte par etter filtrering
allowed_pairs = (
    active_materials[['product_id', 'rm_id']]
    .dropna()
    .drop_duplicates()
)

# 3) Harmoniser typer i data og filtrer med inner merge for å beholde KUN tillatte par
data['product_id'] = pd.to_numeric(data['product_id'], errors='coerce')
data['rm_id'] = pd.to_numeric(data['rm_id'], errors='coerce')

n_before = len(data)
data = data.merge(allowed_pairs.assign(_keep=1), on=['product_id', 'rm_id'], how='inner')
removed_rows = n_before - len(data)

print(f"Kept only non-DELETED pairs: removed {removed_rows} rows (from {n_before} to {len(data)}).")

Kept only non-DELETED pairs: removed 4 rows (from 49223 to 49219).


In [218]:
df = data.copy()

for c in ["date_arrival", "created_date_time"]:
    df[c] = pd.to_datetime(df[c], errors="coerce", utc=True).dt.tz_localize(None)
    
df["lead_time_days"] = (df["date_arrival"] - df["created_date_time"]).dt.days
df["lead_time_days"] = df["lead_time_days"].clip(lower=0)
df["doy_norm"] = df["date_arrival"].dt.dayofyear / 365.0
df["doy_norm_created"] = df["created_date_time"].dt.dayofyear / 365.0

# Numerisk tidsstempel for modellinput
df["created_timestamp"] = df["created_date_time"].view("int64") // 10**9
df.loc[df["created_date_time"].isna(), "created_timestamp"] = np.nan
df["arrival_timestamp"] = df["date_arrival"].view("int64") // 10**9
df.loc[df["date_arrival"].isna(), "arrival_timestamp"] = np.nan




C:\Users\strom\AppData\Local\Temp\ipykernel_21312\3397155453.py:12: FutureWarning: Series.view is deprecated and will be removed in a future version. Use ``astype`` as an alternative to change the dtype.
  df["created_timestamp"] = df["created_date_time"].view("int64") // 10**9
C:\Users\strom\AppData\Local\Temp\ipykernel_21312\3397155453.py:14: FutureWarning: Series.view is deprecated and will be removed in a future version. Use ``astype`` as an alternative to change the dtype.
  df["arrival_timestamp"] = df["date_arrival"].view("int64") // 10**9


In [219]:
# Filtrer til rm_id med aktive leveranser siste to år og hyppig frekvens
if df['date_arrival'].notna().any():
    recent_cutoff = df['date_arrival'].max() - pd.DateOffset(years=2)
    recent_subset = df.loc[df['date_arrival'] >= recent_cutoff].copy()

    if recent_subset.empty:
        print("Ingen leveranser registrert de siste to årene; hopper over aktivitetsfilter.")
    else:
        stats_recent = (
            recent_subset
            .groupby('rm_id')
            .agg(
                deliveries_2y=('date_arrival', 'count'),
                active_months_2y=('date_arrival', lambda s: s.dt.to_period('M').nunique()),
                total_weight_2y=('net_weight', 'sum')
            )
            .reset_index()
        )

        MIN_DELIVERIES_2Y = 20
        MIN_ACTIVE_MONTHS_2Y = 10

        frequent_rm_ids = stats_recent.loc[
            (stats_recent['deliveries_2y'] >= MIN_DELIVERIES_2Y) &
            (stats_recent['active_months_2y'] >= MIN_ACTIVE_MONTHS_2Y),
            'rm_id'
        ]

        before_filter = df['rm_id'].nunique()
        df = df[df['rm_id'].isin(frequent_rm_ids)].copy()
        after_filter = df['rm_id'].nunique()

        print(
            f"Aktivitetsfilter: beholdt {after_filter} rm_id av {before_filter} ",
            f"(>= {MIN_DELIVERIES_2Y} leveranser og >= {MIN_ACTIVE_MONTHS_2Y} aktive måneder siste to år)."
        )
else:
    print("Fant ingen gyldige date_arrival-verdier; hopper over aktivitetsfilter.")

Aktivitetsfilter: beholdt 31 rm_id av 104  (>= 20 leveranser og >= 10 aktive måneder siste to år).


In [220]:
# Features og target - basert på Hannah sine insights
# Legger til 90- og 180-dagers vinduer og andre features

# Sortér data riktig for rolling features
df_sorted = df.sort_values(['rm_id', 'date_arrival']).copy()

# Sett date_arrival som index midlertidig for rolling beregninger
df_sorted = df_sorted.set_index('date_arrival')

# Beregn rolling features for hver rm_id
rolling_features = []

for rm_id in df_sorted['rm_id'].unique():
    rm_data = df_sorted[df_sorted['rm_id'] == rm_id].copy()
    
    # 90-dagers sum
    rm_data['sum_past_90'] = rm_data['net_weight'].rolling('90D').sum()
    
    # 180-dagers sum  
    rm_data['sum_past_180'] = rm_data['net_weight'].rolling('180D').sum()
    
    rolling_features.append(rm_data)

# Kombiner alle rm_id tilbake
df_with_rolling = pd.concat(rolling_features, ignore_index=False)
df_with_rolling = df_with_rolling.reset_index()

# Forhold og differanser mellom vinduer
df_with_rolling['ratio_90_180'] = df_with_rolling['sum_past_90'] / (df_with_rolling['sum_past_180'] + 1e-8)
df_with_rolling['diff_90_180'] = df_with_rolling['sum_past_180'] - df_with_rolling['sum_past_90']

# Ukedag normalisert 
df_with_rolling['weekday'] = df_with_rolling['date_arrival'].dt.weekday / 6

# Kontinuerlig årssyklus (sin/cos for å fange rytme)
df_with_rolling['doy_sin'] = np.sin(2 * np.pi * df_with_rolling['date_arrival'].dt.dayofyear / 365.25)
df_with_rolling['doy_cos'] = np.cos(2 * np.pi * df_with_rolling['date_arrival'].dt.dayofyear / 365.25)

# Måned som kategorisk
df_with_rolling['month'] = df_with_rolling['date_arrival'].dt.month

# Oppdater df med nye features
df = df_with_rolling.copy()

# Fyll NaN verdier i rolling features med 0 (eller forrige verdi)
df['sum_past_90'] = df['sum_past_90'].fillna(0)
df['sum_past_180'] = df['sum_past_180'].fillna(0)
df['ratio_90_180'] = df['ratio_90_180'].fillna(0)
df['diff_90_180'] = df['diff_90_180'].fillna(0)

features = [
    "rm_id",
    "lead_time_days", 
    "sum_past_180",
    "doy_norm"
]

x = df[features].copy()
y = df["net_weight"].values

print(f"Features: {features}")
print(f"Antall features: {len(features)}")
print(f"Shape: {x.shape}")
print(f"Nullverdier i target: {(y == 0).mean():.1%}")
print(f"Rolling features beregnet for {df['rm_id'].nunique()} unike rm_id")

Features: ['rm_id', 'lead_time_days', 'sum_past_180', 'doy_norm']
Antall features: 4
Shape: (38241, 4)
Nullverdier i target: 0.0%
Rolling features beregnet for 31 unike rm_id


In [221]:
# Split data: kun 2018-2024 for trening som Hannah nevnte
train_mask = (df["created_date_time"] >= pd.Timestamp("2018-01-01")) & (df["created_date_time"] < pd.Timestamp("2024-12-31"))
test_mask = (df["date_arrival"] >= pd.Timestamp("2024-01-01")) & (df["date_arrival"] <= pd.Timestamp("2024-05-31"))

X_train, Y_train = x[train_mask], y[train_mask]

y_train_log = np.log1p(Y_train)

print(f"Nullverdier i trening: {(Y_train == 0).mean():.1%}")
print(f"Rader i testsettet: {test_mask.sum()}")

Nullverdier i trening: 0.0%
Rader i testsettet: 2547


In [222]:
# Individuelle modeller per rm_id
from collections import defaultdict
import numpy as np
import pandas as pd

train_df = df.loc[train_mask].copy()
test = df.loc[test_mask].copy()  # sørger for at test-rammen matcher masken
feature_cols_model = [col for col in features if col != "rm_id"]
target_col = "net_weight"

rm_models = {}
fallback_records = []
pred_series = pd.Series(index=test.index, dtype=float)
global_median = float(train_df[target_col].median())
MIN_TRAIN_ROWS = 20

for rm_id, train_grp in train_df.groupby("rm_id"):
    test_idx = test.index[test["rm_id"] == rm_id]
    if test_idx.empty:
        continue

    # Fallback dersom lite historikk eller ingen variasjon
    if len(train_grp) < MIN_TRAIN_ROWS or train_grp[target_col].std() == 0:
        fallback_value = float(train_grp[target_col].median()) if len(train_grp) else global_median
        pred_series.loc[test_idx] = fallback_value
        fallback_records.append({"rm_id": rm_id, "strategy": "median_fallback", "train_rows": int(len(train_grp))})
        continue

    model_rm = lgbm.LGBMRegressor(
        objective="quantile",
        alpha=0.3,
        n_estimators=5000,
        max_depth=6,
        learning_rate=0.04,
        random_state=42,
        verbose=-1
    )

    model_rm.fit(
        train_grp[feature_cols_model],
        np.log1p(train_grp[target_col])
    )

    rm_models[rm_id] = model_rm
    preds_rm = np.expm1(model_rm.predict(test.loc[test_idx, feature_cols_model])*0.975)
    pred_series.loc[test_idx] = preds_rm

# Håndter rm_id som kun finnes i test
missing_idx = pred_series[pred_series.isna()].index
if len(missing_idx) > 0:
    pred_series.loc[missing_idx] = global_median
    for rm_id in test.loc[missing_idx, "rm_id"].unique():
        fallback_records.append({"rm_id": rm_id, "strategy": "global_median", "train_rows": 0})

test["predicted_weight"] = np.clip(pred_series.values, a_min=0.0, a_max=None)

# Oppdater trenings- og testmatriser for videre analyse
X_train = train_df[features].copy()
X_test = test[features].copy()
Y_train = train_df[target_col].values
Y_test = test[target_col].values
y_pred = test["predicted_weight"].values

models_by_rm = rm_models
per_rm_fallback = pd.DataFrame(fallback_records)

preds_2024_rm = (
    test.groupby("rm_id", as_index=False)
        .agg(predicted_weight=("predicted_weight", "sum"))
)

print(f"Trente modeller for {len(models_by_rm)} rm_id. Brukte fallback for {per_rm_fallback['rm_id'].nunique() if not per_rm_fallback.empty else 0} rm_id.")
print("Topp 5 rm_id etter predikert total 2025:")
print(preds_2024_rm.sort_values("predicted_weight", ascending=False).head().to_string(index=False))

if not per_rm_fallback.empty:
    print("\nFallback-oversikt (første 10):")
    print(per_rm_fallback.head(10).to_string(index=False))

Trente modeller for 29 rm_id. Brukte fallback for 0 rm_id.
Topp 5 rm_id etter predikert total 2025:
 rm_id  predicted_weight
3781.0      4.429568e+06
3865.0      3.594270e+06
2130.0      2.315928e+06
3125.0      2.315447e+06
3126.0      2.157839e+06


In [223]:
# Sorter prediksjoner etter rm_id
import numpy as np
import pandas as pd

# Lag en tabell med rm_id, prediksjon og fasit
pred_df = pd.DataFrame({
    "rm_id": X_test["rm_id"].values,
    "y_pred": np.asarray(y_pred),
    "y_true": np.asarray(Y_test),
})

# Sortering etter rm_id
pred_df_sorted = pred_df.sort_values("rm_id", kind="mergesort").reset_index(drop=True)

print(pred_df_sorted.head(10))
print(f"Unique rm_id count: {pred_df_sorted['rm_id'].nunique()}")


    rm_id        y_pred   y_true
0  2129.0   7538.609426  14920.0
1  2129.0   8546.327813  11020.0
2  2129.0   8483.374290  10820.0
3  2129.0   8640.174208  10740.0
4  2129.0   7490.389425  11180.0
5  2129.0   6564.087414  11300.0
6  2130.0  16298.828968  25400.0
7  2130.0  12212.995431  20660.0
8  2130.0  13549.663576  18180.0
9  2130.0  12212.995431  11940.0
Unique rm_id count: 29


In [224]:
# Evaluering av per-rm_id modellene
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
import numpy as np
import pandas as pd

def quantile_loss(y_true, y_hat, q, rm_ids=None):
    """Quantile loss som kan ta hensyn til kumulative prognoser ved å gi rm_ids."""
    if rm_ids is None:
        diff = y_true - y_hat
        return np.mean(np.maximum(q * diff, (q - 1) * diff))
    df = pd.DataFrame({
        "y_true": y_true,
        "y_hat": y_hat,
        "rm_id": rm_ids
    }).sort_values("rm_id")
    total_loss = 0.0
    total_weight = 0
    for rm_id, grp in df.groupby("rm_id"):
        y_t = grp["y_true"].values
        y_p = grp["y_hat"].values
        if len(y_t) > 1:
            y_t_diff = np.diff(np.concatenate([[0], y_t]))
            y_p_diff = np.diff(np.concatenate([[0], y_p]))
            diff = y_t_diff - y_p_diff
            rm_loss = np.mean(np.maximum(q * diff, (q - 1) * diff))
            total_loss += rm_loss * len(y_t_diff)
            total_weight += len(y_t_diff)
        else:
            diff = y_t - y_p
            rm_loss = np.maximum(q * diff, (q - 1) * diff)[0]
            total_loss += rm_loss
            total_weight += 1
    return total_loss / total_weight if total_weight > 0 else 0.0

rm_ids_test = X_test["rm_id"].values if "rm_id" in X_test.columns else test["rm_id"].values

rmse = np.sqrt(mean_squared_error(Y_test, y_pred))
mae = mean_absolute_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
qloss = quantile_loss(Y_test, y_pred, 0.2, rm_ids_test)

baseline_mean = np.full_like(Y_test, Y_train.mean())
baseline_mean_rmse = np.sqrt(mean_squared_error(Y_test, baseline_mean))

residuals = Y_test - y_pred
res_summary = {
    "residual_mean": residuals.mean(),
    "residual_std": residuals.std(),
    "abs_resid_p50": np.percentile(np.abs(residuals), 50),
    "abs_resid_p90": np.percentile(np.abs(residuals), 90),
    "abs_resid_p95": np.percentile(np.abs(residuals), 95),
    "pct_over_predicted": float((residuals < 0).mean())
}

print("=== Modellmetrikker ===")
print(f"RMSE: {rmse:.2f}")
print(f"MAE: {mae:.2f}")
print(f"R2: {r2:.4f}")
print(f"QuantileLoss (q=0.2): {qloss:.4f}")
print(f"BaselineMean_RMSE: {baseline_mean_rmse:.2f}")

print("\n=== Residualanalyse ===")
for k, v in res_summary.items():
    print(f"{k:22s}: {v:.4f}")

# Aggregert feature importance
feature_importance = None
if models_by_rm:
    fi_frames = []
    for rm_id, model in models_by_rm.items():
        if hasattr(model, "feature_importances_"):
            fi_frames.append(pd.DataFrame({
                "rm_id": rm_id,
                "feature": feature_cols_model,
                "importance": model.feature_importances_
            }))
    if fi_frames:
        feature_importance = pd.concat(fi_frames, ignore_index=True)
        agg_importance = (
            feature_importance.groupby("feature")["importance"]
            .mean()
            .sort_values(ascending=False)
        )
        print("\n=== Viktigste features (gjennomsnittlig importance) ===")
        print(agg_importance.head(10).to_string())

def _per_rm_metrics(grp):
    return pd.Series({
        "rows": len(grp),
        "mae_rm": mean_absolute_error(grp[target_col], grp["predicted_weight"]),
        "rmse_rm": np.sqrt(mean_squared_error(grp[target_col], grp["predicted_weight"]))
    })

per_rm_eval = (
    test.groupby("rm_id").apply(_per_rm_metrics)
        .sort_values("rows", ascending=False)
)

print("\n=== Per-rm_id oversikt (topp 10 etter antall observasjoner) ===")
print(per_rm_eval.head(10).to_string())

=== Modellmetrikker ===
RMSE: 6516.10
MAE: 5099.04
R2: 0.4725
QuantileLoss (q=0.2): 2080.8607
BaselineMean_RMSE: 9020.90

=== Residualanalyse ===
residual_mean         : 4642.8512
residual_std          : 4572.0293
abs_resid_p50         : 5180.1522
abs_resid_p90         : 10750.6873
abs_resid_p95         : 13429.1590
pct_over_predicted    : 0.1005

=== Viktigste features (gjennomsnittlig importance) ===
feature
sum_past_180      19126.758621
lead_time_days    18479.275862
doy_norm          17457.310345

=== Per-rm_id oversikt (topp 10 etter antall observasjoner) ===
         rows       mae_rm      rmse_rm
rm_id                                  
3865.0  488.0  5047.979405  6549.186517
3781.0  387.0  6271.665903  7362.133672
2130.0  209.0  6594.013723  7942.042031
3126.0  156.0  6533.032503  7387.103724
2142.0  130.0  2461.241802  4707.046451
3125.0  125.0  5702.500293  5738.057049
3282.0   96.0  5612.719291  5618.727701
3124.0   96.0  5821.021272  5846.536568
3122.0   89.0  5837.754211  

C:\Users\strom\AppData\Local\Temp\ipykernel_21312\2611854612.py:95: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  test.groupby("rm_id").apply(_per_rm_metrics)


In [225]:
# If kaggle_metric.py is in the same directory, import as a module
from kaggle_metric import score, ParticipantVisibleError

# Mapping for 2025 data
mapping = pd.read_csv("data/prediction_mapping.csv", parse_dates=["forecast_start_date", "forecast_end_date"]).copy()

# Robust kolonnenavn-håndtering
if "rm_id" not in mapping.columns:
    for alt in ["material_id", "RM_ID", "rmId"]:
        if alt in mapping.columns:
            mapping = mapping.rename(columns={alt: "rm_id"})
            break

if "ID" not in mapping.columns:
    for alt in ["id", "Id"]:
        if alt in mapping.columns:
            mapping = mapping.rename(columns={alt: "ID"})
            break
    if "ID" not in mapping.columns and "rm_id" in mapping.columns:
        # Fallback: bruk rm_id som ID hvis ID mangler
        mapping["ID"] = mapping["rm_id"].astype(int)

# Sikkerhetssjekk
required_cols = {"ID", "rm_id"}
missing_cols = required_cols - set(mapping.columns)
if missing_cols:
    raise ValueError(f"Mapping mangler kolonner: {missing_cols}. Finnes: {mapping.columns.tolist()}")

# Bygg rm_id-nivå prediksjoner fra 2024-testen
if "X_test" not in globals() or "y_pred" not in globals():
    raise RuntimeError("X_test/y_pred mangler i minnet. Kjør trenings-/prediksjonscellene først.")

preds_2024_rm = pd.DataFrame({
    "rm_id": pd.to_numeric(X_test["rm_id"], errors="coerce"),
    "predicted_weight": np.clip(np.asarray(y_pred), 0, None)
}).groupby("rm_id", as_index=False).agg(predicted_weight=("predicted_weight", "sum"))

# Harmoniser typer før join
mapping["rm_id"] = pd.to_numeric(mapping["rm_id"], errors="coerce")

# Map 2024-prediksjoner til 2025-IDer
submission_2025 = mapping[["ID", "rm_id"]].merge(preds_2024_rm, on="rm_id", how="left")
missing = int(submission_2025["predicted_weight"].isna().sum())
submission_2025["predicted_weight"] = submission_2025["predicted_weight"].fillna(0.0).round(1)
submission_2025 = submission_2025[["ID", "predicted_weight"]].sort_values("ID").reset_index(drop=True)

print(
    f"Totalsammendrag klart i minne: {len(submission_2025)} rader. "
    f"Fylte 0 for {missing} manglende rm_id i mapping."
)
print(submission_2025.head(10).to_string(index=False))

# Backtest for 2024 (beholder, men påvirker ikke daglig submission)
receivals["date_arrival"] = pd.to_datetime(receivals["date_arrival"], errors="coerce", utc=True).dt.tz_localize(None)

start = pd.Timestamp("2024-01-01")
end   = pd.Timestamp("2024-05-31 23:59:59")
solution = (receivals.loc[(receivals["date_arrival"] >= start) & (receivals["date_arrival"] <= end)]
            .groupby("rm_id", as_index=False)
            .agg(weight=("net_weight", "sum"))
           ).rename(columns={"rm_id": "ID"})

rm_col = "rm_id_raw" if "rm_id_raw" in test.columns else "rm_id"

preds = pd.DataFrame({
    "ID": test[rm_col].values,
    "predicted_weight": np.clip(y_pred, 0, None)  
})

submission = preds.groupby("ID", as_index=False).agg(predicted_weight=("predicted_weight", "sum"))

submission = solution[["ID"]].merge(submission, on="ID", how="left")
submission["predicted_weight"] = submission["predicted_weight"].fillna(0.0).round(1)

try:
    final_score = score(solution=solution, submission=submission, row_id_column_name="ID")
    print("Quantile loss (q=0.2) – backtest jan–mai 2024:", final_score)
except ParticipantVisibleError as e:
    print("Scoring feilet:", e)

print(submission.head(10))
print(solution.head(10))

Totalsammendrag klart i minne: 30450 rader. Fylte 0 for 26100 manglende rm_id i mapping.
 ID  predicted_weight
  1               0.0
  2               0.0
  3               0.0
  4               0.0
  5               0.0
  6               0.0
  7               0.0
  8               0.0
  9               0.0
 10               0.0
Quantile loss (q=0.2) – backtest jan–mai 2024: 55209.53043478261
       ID  predicted_weight
0  2124.0               0.0
1  2125.0               0.0
2  2129.0           47263.0
3  2130.0         2315927.9
4  2131.0           49156.0
5  2132.0           39399.5
6  2133.0            8628.7
7  2134.0          167754.0
8  2135.0          197897.9
9  2140.0          811259.9
       ID     weight
0  2124.0     7560.0
1  2125.0    25000.0
2  2129.0    69980.0
3  2130.0  3549704.0
4  2131.0   237344.0
5  2132.0   163266.0
6  2133.0    38740.0
7  2134.0   612846.0
8  2135.0   494030.0
9  2140.0  1046440.0


In [226]:
# Mapping for 2025 data
mapping = pd.read_csv("data/prediction_mapping.csv", parse_dates=["forecast_start_date", "forecast_end_date"]).copy()


In [227]:
# Bygg submission.csv med daglig fordeling per rm_id, én rad per ID (fra mapping)
# Viktig: Vi fordeler TOTAL per rm_id over HELE horisonten først, og så plukker vi
# kumulativt (end-start) for hvert ID-vindu. Dette unngår "alt på dag 1"-problemet.

# Valg for fordeling av daglige shares
going_mode = 'historic'   # 'uniform' | 'historic' | 'blended'
blend_alpha = 0.30        # andel uniform i blend når goiong_mode == 'blended'

# Forutsetninger
if 'preds_2024_rm' not in globals():
    raise RuntimeError("Mangler 'preds_2024_rm' (total per rm_id). Kjør mapping/prediksjonscellen først.")
if 'mapping' not in globals():
    raise RuntimeError("Mangler 'mapping'. Les 'data/prediction_mapping.csv' først.")

# 1) Historiske andeler (shares) per rm_id per dag-av-året (DOY) for jan–mai
receivals_hist = receivals.copy()
receivals_hist["date_arrival"] = pd.to_datetime(receivals_hist["date_arrival"], errors="coerce", utc=True).dt.tz_localize(None)
receivals_hist = receivals_hist.loc[
    (receivals_hist["date_arrival"] < pd.Timestamp("2024-01-01")) &
    (receivals_hist["date_arrival"].dt.month.isin([1,2,3,4,5]))
].copy()
receivals_hist["doy"] = receivals_hist["date_arrival"].dt.dayofyear

by_rm_doy = (
    receivals_hist
    .groupby(["rm_id", "doy"], as_index=False)
    .agg(weight=("net_weight", "sum"))
)
by_rm_tot = by_rm_doy.groupby("rm_id", as_index=False).agg(total=("weight", "sum"))
shares_rm = by_rm_doy.merge(by_rm_tot, on="rm_id", how="left")
shares_rm["share_rm"] = shares_rm["weight"] / shares_rm["total"].replace({0: np.nan})
shares_rm = shares_rm[["rm_id", "doy", "share_rm"]]

# Globale andeler per DOY for fallback
by_doy_glob = receivals_hist.groupby("doy", as_index=False).agg(weight=("net_weight", "sum"))
if by_doy_glob["weight"].sum() <= 0 or by_doy_glob.empty:
    # hvis ingen historikk: antas lik fordeling over jan–mai 2025
    # vi fyller DOY senere basert på horisonten
    by_doy_glob = pd.DataFrame(columns=["doy", "weight"])  

# 2) Definer horisont (datoer) fra mapping
min_start = mapping["forecast_start_date"].min()
max_end   = mapping["forecast_end_date"].max()
all_days = pd.date_range(min_start, max_end, freq='D')
cal = pd.DataFrame({"date": all_days})
cal["doy"] = cal["date"].dt.dayofyear

# Sørg for globale shares for ALLE DOY i horisonten
by_doy_glob = cal[["doy"]].merge(by_doy_glob, on="doy", how="left")
by_doy_glob["weight"] = by_doy_glob["weight"].fillna(1.0)
by_doy_glob["share_glob"] = by_doy_glob["weight"] / by_doy_glob["weight"].sum()
by_doy_glob = by_doy_glob[["doy", "share_glob"]]

# 3) Lag full grid av (rm_id, doy) over hele horisonten
rm_ids = preds_2024_rm["rm_id"].dropna().unique()
rm_grid = pd.DataFrame({"rm_id": np.repeat(rm_ids, len(cal)),
                        "doy": np.tile(cal["doy"].values, len(rm_ids))})

# Knytt rm-spesifikke shares og globale shares
rm_grid = rm_grid.merge(shares_rm, on=["rm_id", "doy"], how="left")
rm_grid = rm_grid.merge(by_doy_glob, on="doy", how="left")

# Velg share: historic vs blended vs uniform
if going_mode == 'uniform':
    rm_grid["share_use"] = 1.0
elif going_mode == 'blended':
    rm_grid["share_use"] = (1.0 - blend_alpha) * rm_grid["share_rm"].fillna(0.0) + blend_alpha * 1.0
else:  # 'historic'
    rm_grid["share_use"] = rm_grid["share_rm"].fillna(rm_grid["share_glob"]).fillna(0.0)

# Normaliser shares innen hver rm_id over HELE horisonten
sum_share = rm_grid.groupby("rm_id")["share_use"].transform("sum").replace({0: np.nan})
rm_grid["share_norm"] = rm_grid["share_use"] / sum_share

# 4) Knytt rm-total og lag daglig vekt per rm_id per dag
rm_grid = rm_grid.merge(preds_2024_rm.rename(columns={"predicted_weight": "rm_total"}), on="rm_id", how="left")
rm_grid["rm_total"] = rm_grid["rm_total"].fillna(0.0)
rm_grid["daily_weight"] = rm_grid["rm_total"] * rm_grid["share_norm"].fillna(0.0)

# Knytt kalenderdato til grid for å få faktiske datoer
rm_daily = rm_grid.merge(cal, on="doy", how="left")[["rm_id", "date", "daily_weight"]]
rm_daily = rm_daily.sort_values(["rm_id", "date"]).reset_index(drop=True)
rm_daily["cumulative_weight"] = rm_daily.groupby("rm_id")["daily_weight"].cumsum()

# 5) Lag submission ved å hente kumulativ vekt for hvert ID-vindu (end - (start-1))
mp = mapping[["ID", "rm_id", "forecast_start_date", "forecast_end_date"]].copy()
mp = mp.merge(rm_daily.rename(columns={"date": "forecast_end_date", "cumulative_weight": "cum_end"}),
              on=["rm_id", "forecast_end_date"], how="left")
# For start-1 dag
mp["start_prev"] = mp["forecast_start_date"] - pd.Timedelta(days=1)
mp = mp.merge(rm_daily.rename(columns={"date": "start_prev", "cumulative_weight": "cum_prev"})[["rm_id", "start_prev", "cum_prev"]],
              on=["rm_id", "start_prev"], how="left")
mp["cum_prev"] = mp["cum_prev"].fillna(0.0)

mp["predicted_weight"] = (mp["cum_end"].fillna(0.0) - mp["cum_prev"]).clip(lower=0)
submission = mp[["ID", "predicted_weight"]].copy()

# 6) Sortér og valider mot sample_submission (én rad per ID)
sample = pd.read_csv("data/sample_submission.csv")
submission = sample[["ID"]].merge(submission, on="ID", how="left")
submission["predicted_weight"] = submission["predicted_weight"].fillna(0.0).round(1)

# Skrive kun submission.csv
submission.to_csv("submission.csv", index=False)

# Hent korrekt total ved å bruke siste kumulative punkt per rm_id
submission_with_meta = (
    mapping[["ID", "rm_id", "forecast_end_date"]]
    .merge(submission, on="ID", how="left")
)
submission_with_meta = submission_with_meta.sort_values(["rm_id", "forecast_end_date"])
total_predicted_weight = submission_with_meta.groupby("rm_id")["predicted_weight"].last().sum()
vekt = submission["predicted_weight"].sum()

print(f"submission.csv skrevet: {len(submission)} rader, unike ID: {submission['ID'].nunique()}")
print(
    f"Total predikert vekt (siste kumulative punkt per rm_id): "
    f"{total_predicted_weight:,.0f} kg ({total_predicted_weight/1e6:.1f} millioner kg)"
)
print(submission.head(5).to_string(index=False))
print(f"Total predikert vekt i submission.csv: {vekt:,.0f} kg ({vekt/1e6:.1f} millioner kg)")


sub2 = pd.read_csv("submission (2).csv").copy()
sub2_total = sub2["predicted_weight"].sum()
print(f"Total predikert vekt i submission (2).csv: {sub2_total:,.0f} kg ({sub2_total/1e6:.1f} millioner kg)")

submission.csv skrevet: 30450 rader, unike ID: 30450
Total predikert vekt (siste kumulative punkt per rm_id): 24,844,237 kg (24.8 millioner kg)
 ID  predicted_weight
  1               0.0
  2               0.0
  3               0.0
  4               0.0
  5               0.0
Total predikert vekt i submission.csv: 1,814,811,298 kg (1814.8 millioner kg)
Total predikert vekt i submission (2).csv: 1,873,363,273 kg (1873.4 millioner kg)
